<a href="https://colab.research.google.com/github/aleja71291/FDL-EA-20252/blob/main/3_Alex_net_ADNI1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Alex-Net
En primera instancia probaremos con una red pre-entrenada.

Descargar y extraer las imágenes y las etiquetas.

In [ ]:
import os
import zipfile
from google.colab import drive

import requests
zip_url = 'https://github.com/aleja71291/FDL-EA-20252/raw/refs/heads/main/content/images_Train_1.zip'
csv_url = 'https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/refs/heads/main/content/y_resampled_df%20(1).csv'

response = requests.get(zip_url)
with open('images_Train_1.zip', 'wb') as f:
    f.write(response.content)

response = requests.get(csv_url)
with open('y_resampled_df%20(1).csv', 'wb') as f:
    f.write(response.content)

with zipfile.ZipFile('images_Train_1.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset_images')

In [ ]:
import pandas as pd

labels_df = pd.read_csv('y_resampled_df%20(1).csv')
label_mapping = {label: idx for idx, label in enumerate(labels_df['CDGLOBAL'].unique())}
labels_df['label'] = labels_df['CDGLOBAL'].map(label_mapping)
labels_df['filename'] = '_' + labels_df.index.astype(str) + '_image.png'

print(labels_df.head())

   CDGLOBAL  label      filename
0         1      0  _0_image.png
1         0      1  _1_image.png
2         0      1  _2_image.png
3         0      1  _3_image.png
4         2      2  _4_image.png


Preparamos las imágenes y etiquetas:

In [ ]:
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class BrainImagesDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.df.iloc[idx]['filename'])
        image = Image.open(img_name).convert('RGB')
        label = self.df.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)
        return image, label

full_dataset = BrainImagesDataset(labels_df, 'dataset_images', transform=transform)


full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)


Descargamos y definimos el modelo. En este caso utilizaremos Alex-net

In [ ]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

num_classes = len(label_mapping)


model = models.alexnet(pretrained=True)

# Congelar capas tempranas
for param in model.parameters():
    param.requires_grad = False

# Reemplazar la última capa
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

# Entrenar la última capa
for param in model.classifier[6].parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Training loop
for epoch in range(50):  # number of epochs
    model.train()
    running_loss = 0.0
    for images, labels in full_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(full_loader)}")

Epoch 1, Loss: 1.3994546662206235
Epoch 2, Loss: 1.2921994095263274
Epoch 3, Loss: 1.2717644650003184
Epoch 4, Loss: 1.2311251422633296
Epoch 5, Loss: 1.2789301120716592
Epoch 6, Loss: 1.21008744187977
Epoch 7, Loss: 1.2575097913327424
Epoch 8, Loss: 1.2222838064898616
Epoch 9, Loss: 1.1796607971191406
Epoch 10, Loss: 1.2056836200797039
Epoch 11, Loss: 1.1958738980085954
Epoch 12, Loss: 1.172796967236892
Epoch 13, Loss: 1.1849233922751055
Epoch 14, Loss: 1.1881525361019631
Epoch 15, Loss: 1.1759018923925317
Epoch 16, Loss: 1.1550789501356042
Epoch 17, Loss: 1.2061976401702217
Epoch 18, Loss: 1.177229360393856
Epoch 19, Loss: 1.1982307278591653
Epoch 20, Loss: 1.183227072591367
Epoch 21, Loss: 1.230036305344623
Epoch 22, Loss: 1.234269608622012
Epoch 23, Loss: 1.196677047273387
Epoch 24, Loss: 1.1979030059731526
Epoch 25, Loss: 1.1732489622157554
Epoch 26, Loss: 1.1980327756508538
Epoch 27, Loss: 1.1968728588974995
Epoch 28, Loss: 1.1496981097304302
Epoch 29, Loss: 1.117223138394563
Epo

In [ ]:
torch.save(model.classifier[6].state_dict(), 'alexnet_classifier_weights.pth')
from google.colab import files
files.download('alexnet_classifier_weights.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Ahora validamos en el set de prueba:

In [ ]:
import requests
zip_url = 'https://github.com/aleja71291/FDL-EA-20252/raw/refs/heads/main/content/images_test_1.zip'
csv_url = 'https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/refs/heads/main/content/y_test%20(1).csv'

response = requests.get(zip_url)
with open('images_test_1.zip', 'wb') as f:
    f.write(response.content)

response = requests.get(csv_url)
with open('y_test%20(1).csv', 'wb') as f:
    f.write(response.content)

with zipfile.ZipFile('images_test_1.zip', 'r') as zip_ref:
    zip_ref.extractall('testdataset_images')

labels_df = pd.read_csv('y_resampled_df%20(1).csv')
label_mapping = {label: idx for idx, label in enumerate(labels_df['CDGLOBAL'].unique())}

test_labels_df = pd.read_csv('y_test%20(1).csv')
test_labels_df['label'] = test_labels_df['CDGLOBAL'].map(label_mapping)
test_labels_df['filename'] = '_' + test_labels_df.index.astype(str) + '_image.png'

print(test_labels_df.head())


   CDGLOBAL  label      filename
0         1      0  _0_image.png
1         1      0  _1_image.png
2         2      2  _2_image.png
3         3      3  _3_image.png
4         1      0  _4_image.png


In [ ]:
# Define transformations for the images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class BrainImagesDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.df.iloc[idx]['filename'])
        image = Image.open(img_name).convert('RGB')
        label = self.df.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)
        return image, label

test_dataset = BrainImagesDataset(test_labels_df, 'testdataset_images', transform=transform)

# Create DataLoader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)


In [ ]:
from sklearn.metrics import precision_score, recall_score
import torch
import torchvision.models as models
import torch.nn as nn # Import nn for nn.Linear

all_labels = []
all_predicted = []

classifier_weights_url = 'https://github.com/aleja71291/FDL-EA-20252/raw/refs/heads/main/content/alexnet_classifier_weights.pth'

classifier_weights_path = 'alexnet_classifier_weights.pth'
!wget {classifier_weights_url} -O {classifier_weights_path}

model = models.alexnet(pretrained=True)

num_classes = len(label_mapping)

model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

model.classifier[6].load_state_dict(torch.load(classifier_weights_path, weights_only=False))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_predicted.extend(predicted.cpu().numpy())

accuracy = sum([pred == true for pred, true in zip(all_predicted, all_labels)]) / len(all_labels)

precision = precision_score(all_labels, all_predicted, average='macro')
recall = recall_score(all_labels, all_predicted, average='macro')

print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")

--2025-11-30 05:16:04--  https://github.com/aleja71291/FDL-EA-20252/raw/refs/heads/main/content/alexnet_classifier_weights.pth
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/refs/heads/main/content/alexnet_classifier_weights.pth [following]
--2025-11-30 05:16:04--  https://raw.githubusercontent.com/aleja71291/FDL-EA-20252/refs/heads/main/content/alexnet_classifier_weights.pth
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 67709 (66K) [application/octet-stream]
Saving to: ‘alexnet_classifier_weights.pth’

alexnet_classifier_ 100%[===================>]  66.12K  --.-KB/s  

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Test Accuracy: 32.43%
Precision: 0.081
Recall: 0.250


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
